# 04 · Hỏi đáp trích xuất (Extractive QA)

Cho **đoạn văn** + **câu hỏi** → tìm **đoạn chữ** trong văn bản là câu trả lời.

```
Đoạn văn : "Lan sinh năm 2005 tại Huế."
Câu hỏi  : "Lan sinh ở đâu?"
Trả lời  : "Huế"   (model dự đoán vị trí BẮT ĐẦU và KẾT THÚC)
```

Lý thuyết: xem `knowledge/04_hoi_dap.md`

In [ ]:
# !pip install -U transformers datasets accelerate

## 0. Import

In [ ]:
import string
import unicodedata
from collections import Counter

import torch
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Cấu hình

In [ ]:
# Dùng model ĐÃ học QA sẵn (squad2) -> điểm cao hơn rất nhiều so với model gốc
MODEL_NAME = "deepset/xlm-roberta-base-squad2"

MAX_LEN = 384     # độ dài tối đa (câu hỏi + đoạn văn)
STRIDE = 128      # phần chồng lấn khi cắt đoạn văn dài

## 2. Đọc dữ liệu
Định dạng chuẩn SQuAD: `id`, `context`, `question`, `answers = {"text": [...], "answer_start": [...]}`.
Câu không có đáp án: `answers["text"] = []`.

In [ ]:
data = load_dataset("rajpurkar/squad_v2")          # tiếng Việt: thử "taidng/UIT-ViQuAD2.0"

train_data = data["train"].select(range(20000))   # lấy bớt cho nhanh
valid_data = data["validation"].select(range(1000))

train_data[0]

## 3. Chuẩn bị dữ liệu train

Đoạn văn dài hơn `MAX_LEN` được cắt thành nhiều cửa sổ chồng nhau:
```
đoạn văn: [=========================================]
cửa sổ 1: [==============]
cửa sổ 2:           [==============]       (chồng nhau STRIDE token)
cửa sổ 3:                     [==============]
```
Với mỗi cửa sổ, tìm vị trí token bắt đầu/kết thúc của câu trả lời.
Nếu câu trả lời không nằm trong cửa sổ → nhãn = vị trí 0 (token `<s>`, nghĩa là "không có đáp án").

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def find_token_positions(offsets, sequence_ids, answer_start, answer_end):
    # vị trí token đầu tiên và cuối cùng của đoạn văn
    context_start = sequence_ids.index(1)
    context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

    # câu trả lời không nằm trọn trong cửa sổ này
    if offsets[context_start][0] > answer_start or offsets[context_end][1] < answer_end:
        return 0, 0

    start_token = context_start
    while start_token <= context_end and offsets[start_token][0] <= answer_start:
        start_token += 1
    start_token -= 1

    end_token = context_end
    while end_token >= context_start and offsets[end_token][1] >= answer_end:
        end_token -= 1
    end_token += 1

    return start_token, end_token

In [ ]:
def prepare_train(batch):
    encoded = tokenizer(
        batch["question"],
        batch["context"],
        truncation="only_second",          # chỉ cắt đoạn văn, không cắt câu hỏi
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,     # tạo nhiều cửa sổ
        return_offsets_mapping=True,        # vị trí ký tự của từng token
        padding="max_length",
    )

    window_to_example = encoded.pop("overflow_to_sample_mapping")
    all_offsets = encoded.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(all_offsets):
        answers = batch["answers"][window_to_example[i]]

        if len(answers["text"]) == 0:       # câu hỏi không có đáp án
            start_positions.append(0)
            end_positions.append(0)
            continue

        answer_start = answers["answer_start"][0]
        answer_end = answer_start + len(answers["text"][0])
        start, end = find_token_positions(offsets, encoded.sequence_ids(i), answer_start, answer_end)
        start_positions.append(start)
        end_positions.append(end)

    encoded["start_positions"] = start_positions
    encoded["end_positions"] = end_positions
    return encoded

In [ ]:
train_ds = train_data.map(prepare_train, batched=True, remove_columns=train_data.column_names)
print(len(train_data), "câu hỏi ->", len(train_ds), "cửa sổ")

## 4. Hàm chấm điểm EM và F1

In [ ]:
def normalize_answer(text):
    text = unicodedata.normalize("NFC", text.lower())
    text = "".join(ch for ch in text if ch not in string.punctuation)
    return " ".join(text.split())


def exact_match(prediction, truth):
    return float(normalize_answer(prediction) == normalize_answer(truth))


def f1(prediction, truth):
    pred_words = normalize_answer(prediction).split()
    true_words = normalize_answer(truth).split()
    if len(pred_words) == 0 or len(true_words) == 0:
        return float(pred_words == true_words)

    common = Counter(pred_words) & Counter(true_words)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_words)
    recall = num_same / len(true_words)
    return 2 * precision * recall / (precision + recall)

Dùng `pipeline` để dự đoán: nó tự cắt cửa sổ, tìm đoạn tốt nhất và xử lý câu không có đáp án.

In [ ]:
from transformers import pipeline


def evaluate(model):
    qa = pipeline("question-answering", model=model, tokenizer=tokenizer, device=0 if device == "cuda" else -1)
    outputs = qa(
        question=valid_data["question"],
        context=valid_data["context"],
        handle_impossible_answer=True,      # cho phép trả lời rỗng
        max_seq_len=MAX_LEN,
        doc_stride=STRIDE,
        batch_size=32,
    )

    total_em = 0
    total_f1 = 0
    for output, answers in zip(outputs, valid_data["answers"]):
        truths = answers["text"] if len(answers["text"]) > 0 else [""]
        total_em += max(exact_match(output["answer"], t) for t in truths)
        total_f1 += max(f1(output["answer"], t) for t in truths)

    n = len(valid_data)
    return {"EM": round(100 * total_em / n, 2), "F1": round(100 * total_f1 / n, 2)}

## 5. Điểm trước khi fine-tune

In [ ]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
print(evaluate(model))

## 6. Fine-tune

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir="qa_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=default_data_collator,
)
trainer.train()

In [ ]:
print(evaluate(trainer.model))

## 7. Dự đoán và lưu bài nộp

In [ ]:
import pandas as pd

qa = pipeline("question-answering", model=trainer.model, tokenizer=tokenizer, device=0 if device == "cuda" else -1)

test_data = valid_data      # khi thi: thay bằng dữ liệu test của đề
outputs = qa(
    question=test_data["question"],
    context=test_data["context"],
    handle_impossible_answer=True,
    batch_size=32,
)

submission = pd.DataFrame({"id": test_data["id"], "answer": [o["answer"] for o in outputs]})
submission.to_csv("submission.csv", index=False)
submission.head()

---
## Nâng cao
- `deepset/xlm-roberta-large-squad2`, lr = 1e-5 → thường +3–6 F1.
- Nếu dữ liệu đề không có câu "không đáp án": bỏ `handle_impossible_answer=True`.
- Xem các câu F1 = 0 để tìm lỗi (khoảng trắng, dấu câu, đáp án quá dài).